Importation des bibliothèques

In [29]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split
from tqdm import tqdm # pour la progression 
import time #pour le temps de calcul

In [30]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [31]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                 #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),  # Moyenne RGB
                         (0.5, 0.5, 0.5))  # Écart-type RGB; pixels deviennent centrés autour de 0
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [32]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x


à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [33]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.1, p2=0.1, p3=0.1, p4=0.1):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or [] # si None, aucune couche à masquer
        self.p1 = p1  # dropout sur conv1 output
        self.p2 = p2  # dropout sur conv2 output
        self.p3 = p3  # dropout sur conv3 output
        self.p4 = p4  # dropout sur fc1 output
        # pas de dropout sur la dernière couche
        self.ps = {'conv1': p1, 'conv2': p2, 'conv3': p3, 'fc1': p4}
    
    def dropout_mask(self, x, p, active=True):
        if not self.training or p == 0.0 or not active:
            return torch.ones_like(x) #conserve la dimension de la couche
        mask = (torch.rand_like(x) > p).float() / (1 - p)
        return mask

    def forward(self, x):
        x = F.relu(self.model.conv1(x))
        x = self.dropout_mask(x, self.ps['conv1'], 'conv1' in self.mc_layers) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv2(x))
        x = self.dropout_mask(x, self.ps['conv2'], 'conv2' in self.mc_layers) * x
        x = self.model.pool(x)

        x = F.relu(self.model.conv3(x))
        x = self.dropout_mask(x, self.ps['conv3'], 'conv3' in self.mc_layers) * x
        x = self.model.pool(x)

        x = x.view(x.size(0), -1) 
        x = F.relu(self.model.fc1(x))
        x = self.dropout_mask(x, self.ps['fc1'], 'fc1' in self.mc_layers) * x
        x = self.model.fc2(x)
        return x  # pas encore normalisé
    
    # x = x.view ... 
    # x = self.dropout_mask
    # x = F.relu(self.model.fc1(x))
    # x = self.model.fc2(x)

    # faire pareil pour chaque couche concernée, même celles pas concernées, faire le dropout avant relu (pour faire comme pytorch, masque les arêtes entrantes)
    # dans le nn.sequential, prend effet que si model.train()

In [34]:
#Avant ReLU
class CNN_MCdropout_beforeReLU(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.1, p2=0.1, p3=0.1, p4=0.1):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []
        self.ps = {'conv1': p1, 'conv2': p2, 'conv3': p3, 'fc1': p4}

    def dropout_mask(self, x, p, active=True):
        if not self.training or p == 0.0 or not active:
            return torch.ones_like(x)
        mask = (torch.rand_like(x) > p).float() / (1 - p)
        return mask

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.dropout_mask(x, self.ps['conv1'], 'conv1' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.dropout_mask(x, self.ps['conv2'], 'conv2' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.dropout_mask(x, self.ps['conv3'], 'conv3' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.dropout_mask(x, self.ps['fc1'], 'fc1' in self.mc_layers) * x
        x = F.relu(x)
        x = self.model.fc2(x)
        return x


In [35]:
#Pytorch
class CNN_MCdropout_torch(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.1, p2=0.1, p3=0.1, p4=0.1):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []

        # On wrappe les couches où on veut du dropout
        if 'conv1' in self.mc_layers:
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU(), nn.Dropout(p1))
        else:
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU())

        if 'conv2' in self.mc_layers:
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU(), nn.Dropout(p2))
        else:
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU())

        if 'conv3' in self.mc_layers:
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU(), nn.Dropout(p3))
        else:
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU())

        if 'fc1' in self.mc_layers:
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU(), nn.Dropout(p4))
        else:
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU())

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.model.fc2(x)
        return x


Fonction d'entraînement et test

In [36]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

train_subset, val_subset = random_split(trainset, [train_size, val_size]) # divise le dataset de manière aléatoire en 90/10

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

Fonction d'évaluation (avant entraînement)

In [37]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [38]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Demande à l'utilisateur quelles couches doivent subir le dropout : ou alors il amène sa liste
user_layers = input(
    "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
    "(choisissez parmi conv1, conv2, conv3, fc1, séparées par des virgules) : ")
mc_layers = [layer.strip() for layer in user_layers.split(',') if layer.strip() in ['conv1','conv2','conv3','fc1']]

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

model = CNN_MCdropout(base_model, mc_layers=mc_layers, p1=0.1, p2=0.1, p3=0.1, p4=0.1).to(device)
model_beforeReLU = CNN_MCdropout_beforeReLU(base_model, mc_layers=mc_layers, p1=0.1, p2=0.1, p3=0.1, p4=0.1).to(device)
model_torch = CNN_MCdropout_torch(base_model, mc_layers=mc_layers, p1=0.1, p2=0.1, p3=0.1, p4=0.1).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")
test_loss2, test_acc2 = evaluate(model_beforeReLU, testloader, device)
print(f"Final Test Loss: {test_loss2:.4f} - Test Acc: {test_acc2:.4f}")
test_loss3, test_acc3 = evaluate(model_torch, testloader, device)
print(f"Final Test Loss: {test_loss3:.4f} - Test Acc: {test_acc3:.4f}")

Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267
Final Test Loss: 0.8186 - Test Acc: 0.7267
Final Test Loss: 0.8186 - Test Acc: 0.7267


une fois que j'entraîne le modèle je sauvegarde les poids au format pickle, puis model_orig.loadstatedict (?) puis mettre le chemin de mes poids

MC Dropout prédiction

In [40]:
def mc_predict_mean_probs(model, X, T=1000, verbose=True):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    start_time = time.time() 
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    elapsed = time.time() - start_time
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    if verbose:
        print(f"Temps total: {elapsed:.2f} s  |  Temps moyen par passe: {elapsed/T:.4f} s")
    return probs_mc.mean(0), elapsed                        

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [41]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs, t1 = mc_predict_mean_probs(model, X, T=1000, verbose=True)
probs2, t2 = mc_predict_mean_probs(model_beforeReLU, X, T=1000, verbose=True)
probs3, t3 = mc_predict_mean_probs(model_torch, X, T=1000, verbose=True)
Y_hat = probs.argmax(1)
Y_hat2 = probs2.argmax(1)
Y_hat3 = probs3.argmax(1)

print("Classes vraies       :", Y.tolist())
print(f"After ReLU   (t={t1:.2f}s): {Y_hat.tolist()}")
print(f"Before ReLU  (t={t2:.2f}s): {Y_hat2.tolist()}")
print(f"Torch Dropout (t={t3:.2f}s): {Y_hat3.tolist()}")


100%|██████████| 1000/1000 [01:22<00:00, 12.14it/s]


Temps total: 82.39 s  |  Temps moyen par passe: 0.0824 s


100%|██████████| 1000/1000 [01:28<00:00, 11.33it/s]


Temps total: 88.29 s  |  Temps moyen par passe: 0.0883 s


100%|██████████| 1000/1000 [00:57<00:00, 17.38it/s]

Temps total: 57.52 s  |  Temps moyen par passe: 0.0575 s
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]
After ReLU   (t=82.39s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 9, 4, 2, 4, 0, 9, 6, 6, 5, 4, 3, 9, 3, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 5, 9, 7, 6, 9, 8, 6, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 6, 1, 0, 5, 9, 8, 6, 8, 8, 0, 2, 0, 3, 3, 8, 8, 1, 1, 7, 2, 9, 2, 8, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 3, 5, 5, 6, 3, 1, 1, 2, 6, 8, 5, 6, 0, 2, 2, 9, 3, 0, 4, 3, 5, 8, 3, 1, 2, 8, 0, 8, 3]
Before ReLU  (t=88.29s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 9, 4, 2, 4, 0, 9, 6, 6, 5, 4, 

rajouter en argument de generate_mc_outputs measure = variance (en gros on laisse à l'utilisateur le choix de la mesure d'incertitude)

In [ ]:
# def generate_mc_outputs(model, X, T=1000, metrics="mc_estimate", labels=None, verbose=True):
#     model.train()  # dropout actif en inférence
#     outputs = []
#     start_forward = time.time()
#     with torch.no_grad():
#         for _ in tqdm(range(T), disable=not verbose):
#             out = model(X)                      
#             outputs.append(out.unsqueeze(0))    
#     elapsed_forward = time.time() - start_forward
    
#     outputs = torch.cat(outputs, dim=0)         
#     all_probs   = torch.softmax(outputs, dim=2)  
#     mean_probs  = all_probs.mean(dim=0)          # moyenne des softmax
#     var_pred  = outputs.var(dim=0)                # variance des logits
   
#     results = {}
#     elapsed_metrics = {}
#     for metric in metrics:
#         start_metric = time.time()
#         if metric == "mc_estimate":
#             results[metric] = mean_probs
#         elif metric == "variance":
#             results[metric] = var_pred
#         elif metric == "predictive_entropy":
#             results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1) # +1e-12 pour éviter log(0)
#         elif metric == "relative_norm":
#             if labels is None:
#                 raise ValueError("labels must be provided for relative_norm metric")
#             labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
#             diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)   # ||ŷ̄ - Y||
#             denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
#             results[metric] = diff_norm / (denom + 1e-12)                  
#         else:
#             raise ValueError(f"Metric {metric} non reconnue")
#         elapsed_metrics[metric] = time.time() - start_metric
#     model.eval()  # remettre le modèle en mode eval à la fin

#     if verbose:
#         print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
#         for m, t in elapsed_metrics.items():
#             print(f"Temps calcul métrique '{m}': {t:.6f}s")

#     return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


In [50]:
def generate_mc_outputs(model, X, T=1000, metrics=["mc_estimate"], labels=None, verbose=True):
    model.train()
    outputs = []

    # --- forward pass ---
    start_forward = time.time()
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    elapsed_forward = time.time() - start_forward

    outputs = torch.cat(outputs, dim=0)
    results = {}
    elapsed_metrics = {}

    # --- calcul des métriques avec chrono réel ---
    for metric in metrics:
        start_metric = time.time()
        if metric == "mc_estimate":
            all_probs = torch.softmax(outputs, dim=2)
            results[metric] = all_probs.mean(dim=0)
        elif metric == "variance":
            results[metric] = outputs.var(dim=0)
        elif metric == "predictive_entropy":
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)
        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels must be fournis pour relative_norm")
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            results[metric] = diff_norm / (denom + 1e-12)
        else:
            raise ValueError(f"Métrique {metric} non reconnue")
        elapsed_metrics[metric] = time.time() - start_metric

    model.eval()

    if verbose:
        print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
        for m, t in elapsed_metrics.items():
            print(f"Temps calcul métrique '{m}': {t:.6f}s")

    return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


le mc estimate n'est pas une mesure d'incertitude, sert pour construire une autre variable

voir si je peux renvoyer différentes métriques à la dmd de l'utilisateur

Métriques avec la méthode after ReLU

In [51]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 63.59 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.004075 s
Résultat : tensor([[1.0771e-03, 8.4030e-04, 1.7675e-03,  ..., 5.9383e-04, 1.9692e-02,
         2.5078e-03],
        [1.0429e-03, 3.9508e-01, 3.2074e-06,  ..., 4.1113e-08, 6.0141e-01,
         2.4574e-03],
        [1.2589e-01, 1.4259e-01, 9.1100e-04,  ..., 1.5212e-03, 6.8499e-01,
         3.4733e-02],
        ...,
        [4.7774e-01, 1.7563e-01, 1.9764e-01,  ..., 1.2460e-03, 7.3869e-02,
         5.3193e-02],
        [1.7790e-01, 5.8169e-04, 1.0022e-02,  ..., 9.8638e-05, 7.4353e-01,
         1.5534e-03],
        [2.1490e-04, 9.5389e-07, 1.8161e-03,  ..., 2.9551e-03, 1.0192e-05,
         4.1481e-04]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.004108 s
Résultat : tensor([[ 6.2172,  4.6711,  4.2334,  ...,  5.2087,  6.7728,  8.4011],
       

Métriques avec la méthode before ReLU

In [52]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model_beforeReLU, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 66.77 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.010801 s
Résultat : tensor([[1.8949e-03, 1.0070e-03, 2.3813e-03,  ..., 6.4602e-04, 1.8823e-02,
         1.7693e-03],
        [4.4956e-04, 3.9027e-01, 1.0905e-05,  ..., 5.4335e-07, 6.0618e-01,
         3.0673e-03],
        [1.3898e-01, 1.4320e-01, 1.7119e-03,  ..., 1.2367e-03, 6.7365e-01,
         3.1663e-02],
        ...,
        [4.7665e-01, 1.8088e-01, 1.9447e-01,  ..., 1.4776e-03, 7.8408e-02,
         5.1218e-02],
        [1.6099e-01, 1.5302e-03, 1.1224e-02,  ..., 6.2783e-05, 7.5931e-01,
         1.0472e-03],
        [8.1315e-04, 8.3491e-06, 1.3170e-03,  ..., 2.1614e-03, 8.8257e-06,
         3.0881e-04]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.007415 s
Résultat : tensor([[ 6.4512,  4.3668,  4.4286,  ...,  5.3716,  6.4199,  8.0830],
       

Métriques avec la méthode PyTorch

In [53]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model_torch, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 49.41 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.018101 s
Résultat : tensor([[5.7134e-04, 3.6601e-04, 1.0604e-03,  ..., 4.2726e-04, 7.8957e-03,
         4.1908e-04],
        [2.2872e-04, 2.9201e-01, 2.3708e-06,  ..., 2.7007e-08, 7.0692e-01,
         8.3804e-04],
        [1.3420e-01, 7.6464e-02, 1.1560e-03,  ..., 1.5563e-03, 7.4809e-01,
         2.4206e-02],
        ...,
        [6.0822e-01, 1.2474e-01, 1.7750e-01,  ..., 7.6342e-04, 4.6939e-02,
         3.1481e-02],
        [1.7574e-01, 3.9329e-04, 9.1714e-03,  ..., 8.2784e-05, 7.4409e-01,
         1.2433e-03],
        [1.9728e-04, 8.0477e-07, 1.0372e-03,  ..., 1.3983e-03, 4.5568e-06,
         2.8146e-04]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.010034 s
Résultat : tensor([[1.9343, 1.4302, 1.3430,  ..., 1.6280, 2.1396, 2.6009],
        [1.57

pour la fonction de confiance, dois-je faire 1/métrique choisie? ou ça marche qu'avec la variance?

oui si c'est dans R+ : 1/ ? exp(-...) si c'est pour avoir entre 0 et 1 ; quitte à normaliser après